<a href="https://colab.research.google.com/github/Sweety771-afk/CSA1613R/blob/main/Assignment_Dwdm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Generate E-Commerce Data
np.random.seed(42)
n = 1000

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Age": np.random.randint(18, 65, n),
    "Gender": np.random.choice(["Male", "Female"], n),
    "Income": np.random.randint(20000, 120000, n),
    "Orders": np.random.randint(1, 20, n),
    "TotalSpent": np.random.randint(500, 100000, n),
    "WebSessions": np.random.randint(1, 50, n),
    "SupportCalls": np.random.randint(0, 10, n),
    "LastPurchaseDays": np.random.randint(1, 365, n),
    "PaymentMethod": np.random.choice(
        ["Card", "UPI", "Net Banking", "COD"], n),
    "Churn": np.random.choice([0, 1], n, p=[0.7, 0.3])
})

# 2. ETL - Extract, Transform, Load

# Extract
df = data.copy()

# Transform
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])
df["PaymentMethod"] = le.fit_transform(df["PaymentMethod"])

# 3. Create Data Warehouse
conn = sqlite3.connect("ecommerce_warehouse.db")
df.to_sql("customer_fact", conn, if_exists="replace", index=False)

print("Data Warehouse Created Successfully")

# 4. OLAP Analysis

print("\nCustomer Summary:")
print(df[["Age", "Orders", "TotalSpent", "WebSessions"]].describe())

print("\nAverage Spending by Gender:")
print(df.groupby("Gender")["TotalSpent"].mean())

print("\nAverage Orders by Payment Method:")
print(df.groupby("PaymentMethod")["Orders"].mean())

# 5. Prepare Data for Data Mining

X = df.drop(["CustomerID", "Churn"], axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

# 7. Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

# 8. SVM
svm = SVC()
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

# 9. Model Comparison
results = pd.DataFrame({
    "Model": ["Decision Tree", "Naive Bayes", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, nb_pred),
        accuracy_score(y_test, svm_pred)
    ]
})

print("\nModel Comparison:")
print(results)

# 10. Best Model
best = results.loc[results["Accuracy"].idxmax()]
print("\nBest Model:", best["Model"])
print("Accuracy:", round(best["Accuracy"] * 100, 2), "%")

# 11. Classification Report
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, dt_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dt_pred))

conn.close()

Data Warehouse Created Successfully

Customer Summary:
               Age       Orders    TotalSpent  WebSessions
count  1000.000000  1000.000000   1000.000000  1000.000000
mean     40.986000     9.986000  49956.105000    25.236000
std      13.497852     5.516542  29077.010882    13.975107
min      18.000000     1.000000    612.000000     1.000000
25%      29.000000     5.000000  25096.500000    13.000000
50%      42.000000    10.000000  48226.000000    24.000000
75%      52.000000    15.000000  76743.250000    38.000000
max      64.000000    19.000000  99803.000000    49.000000

Average Spending by Gender:
Gender
0    49851.014768
1    50050.806084
Name: TotalSpent, dtype: float64

Average Orders by Payment Method:
PaymentMethod
0     9.414097
1     9.569767
2    10.413934
3    10.476015
Name: Orders, dtype: float64

Model Comparison:
           Model  Accuracy
0  Decision Tree     0.555
1    Naive Bayes     0.705
2            SVM     0.705

Best Model: Naive Bayes
Accuracy: 70.5 %

D